# Train the remote-sensing captioner (VRSBench)

**Goal:** given ONE satellite/aerial image, write a detailed, human-readable description of it -- the "explain / describe this image" capability the app was missing (its VQA specialist answers in one or two words by construction, and its object detector only knows a handful of object types). The problem statement names VRSBench for evaluating single-image captioning; this notebook is the remote-sensing adaptation that makes the description come from a domain-trained model instead of a generic VLM.

**Data:** [VRSBench](https://huggingface.co/datasets/xiang709/VRSBench) (CC-BY-4.0): 20,264 train images with one detailed caption each (mean 53 words), and a held-out eval set of 9,350 images with reference captions. Downloaded inside the notebook from the Hugging Face CDN (Kaggle's network is fast; a home connection isn't) -- ~13GB of zips that are **read in place** rather than extracted.

**Model:** [SmolVLM-500M-Instruct](https://huggingface.co/HuggingFaceTB/SmolVLM-500M-Instruct) (Apache-2.0, ungated, 507M parameters). Chosen because it has to run *next to* the other specialists on a 16GB laptop (about 1GB in half precision; the PaliGemma VQA model alone is ~6GB), and because a T4 has no bf16, so fp16 training has to be safe -- a 500M SigLIP+Llama-style model is. Fine-tuned with LoRA on the language model plus a trainable vision-to-text connector; the vision encoder stays frozen.

## What was checked before writing this (so the cells below aren't guesses)

* **The processor's default is wasteful.** `do_image_splitting=True` upscales a 512x512 image into a 4x4 grid of tiles = **1,088 image tokens** (the zero-shot run took ~6 s per image and 1,142 prompt tokens). With splitting off the same image is **64 tokens** (81-token prompt). Everything here -- training and inference -- uses splitting off, at VRSBench's native 512x512.
* **The zero-shot base model is a poor remote-sensing describer** (run locally on five real Esri scenes): it called a container port "a bridge spanning a wide body of water" and Iowa farmland "a densely populated city". That is the gap fine-tuning has to close, and it is why this notebook measures a zero-shot baseline on the same images as the fine-tuned model.
* **Most training captions carry provenance boilerplate.** 75.6% mention Google Earth ("The high-resolution image sourced from GoogleEarth shows..."), a further ~14% name a sensor ("captured by GF with medium resolution" -- GF/JL are the Gaofen and Jilin satellites), and 2,164 claim "medium resolution". A model trained on that would state a source and a resolution for every image it is shown -- pure hallucination -- so all of it is stripped from the training targets *and* from the evaluation references (same rules, so the comparison stays fair).
* **No geometric augmentation.** Captions say "on the left side", "top-right corner": a flip or rotation would silently make the label false.
* **Loss is on the answer only.** The prompt and the 64 image tokens are masked; the end-of-utterance token is a label so the model learns to stop.

## Metrics

BLEU-1..4, ROUGE-L and CIDEr on 1,000 held-out images from VRSBench's official eval set (one reference each, so absolute numbers are low for every model -- compare fine-tuned against zero-shot on the same images), plus mean generated length against the reference length. Numbers here say the *style and vocabulary* are right; they do not prove factual accuracy, which the app checks separately on real scenes.

Run cells top to bottom. Kaggle: enable **Internet** and a **GPU**.

## 0. GPU compatibility check

**Found live, via an actual failed run on Kaggle**: this session's preinstalled PyTorch build
(2.10.0+cu128) only supports CUDA compute capabilities sm_70 and up -- it silently drops support
for the Pascal-generation **P100** (sm_60), one of the two GPU types Kaggle itself still offers
here (T4x2 or P100, per the note below). Landing on a P100 crashed training ~40 seconds in with
`CUDA error: no kernel image is available for execution on the device` -- a real run, not a
hypothetical. The cell below detects the actual GPU via `nvidia-smi` (no torch import needed yet,
so this runs before torch's own compute-capability list is fixed for the process) and reinstalls
a CUDA 11.8 build if the assigned GPU isn't in the preinstalled build's supported list -- CUDA 11.8
wheels cover Pascal through Hopper, so this works regardless of which GPU Kaggle happens to assign.

In [ ]:
import subprocess, sys

try:
    cc_raw = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True
    ).strip().splitlines()[0]
    major, minor = cc_raw.split(".")
    needed_sm = f"sm_{major}{minor}"
except Exception as e:
    needed_sm = None
    print(f"Could not query GPU compute capability via nvidia-smi ({e}) -- skipping the compatibility check.")

if needed_sm:
    # Check the INSTALLED build's supported architectures in a SEPARATE PROCESS, not an in-process
    # `import torch` -- Python caches imports in sys.modules, so even an aliased/deleted in-process
    # import here would make a LATER `import torch` in the next cell silently return the stale
    # cached module instead of a fresh one. Confirmed live: an earlier version of this cell did
    # `import torch as _torch_probe`, and the kernel died ~80s after reinstalling -- the old torch
    # stayed resident in this process while its .so files got replaced out from under it on disk.
    check = subprocess.run(
        [sys.executable, "-c",
         "import torch; print(' '.join(torch.cuda.get_arch_list()) if torch.cuda.is_available() else '')"],
        capture_output=True, text=True,
    )
    supported = check.stdout.split()
    if needed_sm not in supported:
        print(f"GPU needs {needed_sm}, not in the preinstalled torch build's supported list "
              f"{supported} -- reinstalling a CUDA 11.8 build (covers Pascal through Hopper)...")
        # Uninstall the whole torch/torchvision/torchaudio trio first, then install all three
        # together from the SAME cu118 index in one resolution -- reinstalling `torch` alone
        # left mismatched torchvision/nccl versions behind, which crashed two live runs with
        # unrelated-looking errors (undefined symbol ncclCommShrink; aten.OpaqueObject not
        # registered) that were actually both this same root cause from a different angle.
        subprocess.run(["pip", "uninstall", "-y", "-q", "torch", "torchvision", "torchaudio"], check=True)
        subprocess.run(
            ["pip", "install", "-q", "torch", "torchvision", "torchaudio",
             "--index-url", "https://download.pytorch.org/whl/cu118"],
            check=True,
        )
        print("Reinstalled. The check above ran in a subprocess, so THIS process has never "
              "imported torch itself -- the next cell's `import torch` will be a genuinely fresh "
              "import, not a cached one.")
    else:
        print(f"GPU compute capability {needed_sm} already supported by the preinstalled build.")

In [ ]:
import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(" -", torch.cuda.get_device_name(i))

## 1. Setup

In [ ]:
!pip install -q peft pycocoevalcap hf_transfer
# Kaggle's image ships an old torchao (0.10) that the freshly installed peft refuses to import alongside
# ("Found an incompatible version of torchao ... only versions above 0.16.0") -- the first run died on
# exactly this, at get_peft_model, after 20 minutes of downloads. Nothing here uses torchao.
!pip uninstall -y -q torchao

In [ ]:
import os, sys, json, re, math, time, random, zipfile, io, shutil, subprocess, collections
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"  # set before huggingface_hub is imported (see the download cell)

# Local smoke-test hooks -- never set on Kaggle. CAP_DATA_ROOT points at a folder holding the four
# VRSBench files (real captions, a handful of real images in mini zips) so the whole notebook can be
# run end to end on a laptop before any GPU time is spent.
SMOKE_TEST = bool(os.environ.get("CAP_SMOKE_TEST"))
DATA_ROOT = os.environ.get("CAP_DATA_ROOT")
MODEL_ID = os.environ.get("CAP_MODEL_ID", "HuggingFaceTB/SmolVLM-500M-Instruct")

PROMPT = "Describe the image in detail."
SEED = 0
N_VAL, N_TEST = (8, 8) if SMOKE_TEST else (300, 1000)
EPOCHS = 1 if SMOKE_TEST else 3
BATCH = 2 if SMOKE_TEST else 32
WORKERS = 0 if SMOKE_TEST else 4
LR = 2e-4
MAX_NEW_TOKENS = 24 if SMOKE_TEST else 128

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| smoke test:", SMOKE_TEST, "| model:", MODEL_ID)

## 2. Model input, generation and metrics

`chat()` builds exactly the prompt the app will use at inference. `collate` masks everything up to and including the last `Assistant:` marker, so only the caption (and the end-of-utterance token) is trained on. These helpers need no data, so they come first -- the pre-flight below runs them before anything is downloaded.

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText

processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.image_processor.do_image_splitting = False  # 64 image tokens per 512px image, not 1,088
tokenizer = processor.tokenizer
ASSISTANT_IDS = tokenizer.encode("Assistant:", add_special_tokens=False)

def chat(caption=None):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT}]}]
    if caption is None:
        return processor.apply_chat_template(messages, add_generation_prompt=True)
    messages.append({"role": "assistant", "content": [{"type": "text", "text": caption}]})
    return processor.apply_chat_template(messages, add_generation_prompt=False).rstrip("\n")  # ends on <end_of_utterance>

class ZipImages:
    """Reads images straight out of a zip. The handle is opened lazily, per process, so DataLoader
    workers each get their own."""
    def __init__(self, path, folder):
        self.path, self.folder, self._zip = path, folder, None
    def load(self, name):
        if self._zip is None:
            self._zip = zipfile.ZipFile(self.path)
        return Image.open(io.BytesIO(self._zip.read(f"{self.folder}/{name}"))).convert("RGB")
    def __getstate__(self):
        state = self.__dict__.copy(); state["_zip"] = None
        return state

class Captions(Dataset):
    def __init__(self, entries, images):
        self.entries, self.images = entries, images
    def __len__(self):
        return len(self.entries)
    def __getitem__(self, i):
        name, caption = self.entries[i]
        return self.images.load(name), caption

def collate(batch):
    enc = processor(text=[chat(c) for _, c in batch], images=[[img] for img, _ in batch], return_tensors="pt", padding=True)
    labels = enc["input_ids"].clone()
    k = len(ASSISTANT_IDS)
    for row in range(labels.shape[0]):
        ids = enc["input_ids"][row].tolist()
        start = max(i for i in range(len(ids) - k + 1) if ids[i:i + k] == ASSISTANT_IDS) + k
        labels[row, :start] = -100
    labels[enc["attention_mask"] == 0] = -100
    enc["labels"] = labels
    return enc

from peft import LoraConfig, get_peft_model

def add_lora(model):
    """LoRA on the text model's projections only (the regex skips the SigLIP vision layers, which share
    the projection names), plus the vision-to-text connector trained in full."""
    model = get_peft_model(model, LoraConfig(
        r=32, lora_alpha=64, lora_dropout=0.05,
        target_modules=r".*text_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)",
    ))
    for name, p in model.named_parameters():
        if "connector" in name:
            p.requires_grad = True
    return model

from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider

@torch.no_grad()
def generate(model, entries, images, batch=16, max_new_tokens=MAX_NEW_TOKENS):
    model.eval()
    tokenizer.padding_side = "left"
    prompt = chat(None)
    texts = []
    for i in range(0, len(entries), batch):
        chunk = entries[i:i + batch]
        inputs = processor(text=[prompt] * len(chunk), images=[[images.load(n)] for n, _ in chunk], return_tensors="pt", padding=True).to(DEVICE)
        with torch.autocast("cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.05)
        texts += [t.strip() for t in processor.batch_decode(ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)]
    tokenizer.padding_side = "right"
    return texts

def caption_metrics(references, hypotheses):
    norm = lambda s: " ".join(re.findall(r"[a-z0-9]+", s.lower()))
    gts = {i: [norm(r)] for i, r in enumerate(references)}
    res = {i: [norm(h) or "empty"] for i, h in enumerate(hypotheses)}
    bleu, _ = Bleu(4).compute_score(gts, res, verbose=0)
    rouge, _ = Rouge().compute_score(gts, res)
    cider, _ = Cider().compute_score(gts, res)
    return {"BLEU-1": bleu[0], "BLEU-2": bleu[1], "BLEU-3": bleu[2], "BLEU-4": bleu[3], "ROUGE-L": rouge, "CIDEr": cider,
            "mean_words": float(np.mean([len(h.split()) for h in hypotheses])),
            "reference_mean_words": float(np.mean([len(r.split()) for r in references]))}

## 3. Pre-flight: the whole train / save / reload path, before any download

The first Kaggle run spent 20 minutes downloading 12GB and then died at `get_peft_model` on a library-version clash. This cell runs the *same* functions the real run uses -- LoRA injection, two fp16-autocast optimiser steps with gradient scaling, generation, merge, save and reload of the exported artifact -- on four synthetic images, so an environment problem shows up in the first two minutes instead of after the downloads. It cannot catch data-scale problems (those surface in the first training steps).

In [ ]:
class SyntheticImages:
    """Stands in for ZipImages: deterministic random 512px images."""
    def load(self, name):
        rng = np.random.default_rng(sum(map(ord, name)))
        return Image.fromarray(rng.integers(0, 255, (512, 512, 3), dtype=np.uint8))

def preflight():
    t0 = time.time()
    use_amp = DEVICE == "cuda"
    m = add_lora(AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.float32).to(DEVICE))
    params = [p for p in m.parameters() if p.requires_grad]
    assert params, "no trainable parameters after add_lora()"
    opt = torch.optim.AdamW(params, lr=1e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    images = SyntheticImages()
    batch = collate([(images.load(f"x{i}"), f"The image shows {i + 2} large buildings beside a road on the left side.") for i in range(4)])
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    m.train()
    for _ in range(2):  # the real loop's exact sequence: autocast forward, scaled backward, unscale, clip, step
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            loss = m(**batch).loss
        assert torch.isfinite(loss), f"non-finite loss in the very first steps ({float(loss)}) -- fp16 is overflowing"
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
    out = generate(m, [("a", ""), ("b", "")], images, batch=2, max_new_tokens=8)
    assert len(out) == 2
    merged = m.merge_and_unload().half()
    tmp = "/tmp/preflight_model"
    merged.save_pretrained(tmp, safe_serialization=True); processor.save_pretrained(tmp)
    del m, merged, opt
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    reloaded = AutoModelForImageTextToText.from_pretrained(tmp, dtype=torch.float16).to(DEVICE)
    out = generate(reloaded, [("a", "")], images, batch=1, max_new_tokens=8)
    assert len(out) == 1
    del reloaded
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    shutil.rmtree(tmp, ignore_errors=True)
    print(f"pre-flight OK in {time.time() - t0:.0f}s: LoRA + autocast train steps, generate, merge/save/reload all run in THIS environment")

preflight()

## 4. Get VRSBench

`Images_train.zip` (8.4GB) and `Images_val.zip` (4.0GB) are PNG-in-zip (deflate saves ~6%), so they are opened in place by the dataset below -- no extraction, no doubling of the disk footprint. The first run's plain `wget` managed ~8 MB/s (17 minutes for the train zip); `hf_transfer` (parallel chunks) is tried first, with `wget` as the fallback. Scratch space is `/kaggle/temp` if it exists, else `/tmp` (1.1TB free on the T4 image); either way, *not* `/kaggle/working`, whose contents are saved as this run's output.

In [ ]:
HF = "https://huggingface.co/datasets/xiang709/VRSBench/resolve/main/"
FILES = ["VRSBench_train.json", "VRSBench_EVAL_Cap.json", "Images_train.zip", "Images_val.zip"]
SCRATCH_DIR = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"

def get_file(name):
    if DATA_ROOT:
        return os.path.join(DATA_ROOT, name)
    dest = os.path.join(SCRATCH_DIR, name)
    if os.path.exists(dest):
        return dest
    t0 = time.time()
    try:  # hf_transfer downloads in parallel chunks; a plain wget of the same file managed ~8 MB/s
        from huggingface_hub import hf_hub_download
        hf_hub_download("xiang709/VRSBench", name, repo_type="dataset", local_dir=SCRATCH_DIR)
        how = "hf_transfer"
    except Exception as exc:
        print(f"  hf download failed for {name} ({type(exc).__name__}: {str(exc)[:100]}) -- falling back to wget")
        subprocess.run(["wget", "-q", "-c", HF + name, "-O", dest], check=True)  # -c: resume a dropped connection
        how = "wget"
    size = os.path.getsize(dest)
    print(f"downloaded {name} ({size / 1e9:.2f} GB) in {time.time() - t0:.0f}s via {how} ({size / 1e6 / max(time.time() - t0, 1):.0f} MB/s)", flush=True)
    return dest

if not DATA_ROOT:
    free_gb = shutil.disk_usage(SCRATCH_DIR).free / 1e9
    print(f"free space in {SCRATCH_DIR}: {free_gb:.0f} GB")
    assert free_gb > 16, "need ~13GB for the two image zips plus headroom"

paths = {name: get_file(name) for name in FILES}
for name, p in paths.items():
    print(f"{name:22s} {os.path.getsize(p) / 1e6:10.1f} MB")

## 5. Captions: cleaning, splits and loaders

The cleaning rules were developed against all 20,264 real training captions (18,518 change; rules for Google Earth, the GF/JL satellite names and resolution claims); the `usable` filter then drops the ~8% whose cleaned text is still broken ("The image is a medium-resolution grayscale aerial shot..." with the resolution phrase half-removed, quoted source names, ...) -- 18,626 captions survive. The same rules are applied to the evaluation references (8,500 of 9,350 survive), so training and evaluation see the same style.

Splits: 300 training images held out (never trained on) for validation loss / early stopping, and 1,000 random images from VRSBench's *official* eval set for the reported test metrics.

In [ ]:
_PROV = r"(?:google\s?earth|gf(?:-?\d+)?|jl(?:-?\d+)?|gaofen(?:-?\d+)?|jilin(?:-?\d+)?)(?:\s+(?:source|satellite|imagery|sensor))?"
_SRC = r"(?:sourced\s+from|taken\s+from|captured\s+(?:by|from)|obtained\s+from|acquired\s+(?:by|from)|provided\s+(?:by|from)|courtesy\s+of|generated\s+by|from|via|on|by)"
_RES = r"(?:medium|high|low|moderate|specific|specified|unspecified|unknown|visible|explicit)[- ]?resolution"
_RULES = [
    # "..., sourced from GoogleEarth, ..." / "captured by GF with medium resolution"
    (re.compile(rf",?\s+(?:which\s+is\s+|that\s+is\s+)?{_SRC}\s+{_PROV}(?:\s+(?:with|at|in|of)\s+(?:a\s+)?{_RES})?\s*,?", re.I), " "),
    (re.compile(rf"\b(?:is|was)\s+{_SRC}\s+{_PROV}\s+and\b", re.I), "and"),
    # "the GoogleEarth image" / "the GF grayscale image"
    (re.compile(rf"\b{_PROV}\s+(?=(?:high-resolution\s+)?(?:aerial\s+|satellite\s+|grayscale\s+|color\s+)?(?:image|view|photo|imagery))", re.I), ""),
    (re.compile(rf"\b{_PROV}\b", re.I), "the imagery"),
    # resolution claims the model could never verify
    (re.compile(rf"\s*,?\s+(?:with|at|in|of)\s+(?:a\s+|an\s+)?{_RES}", re.I), ""),
    (re.compile(rf"\b{_RES}\s+(?=(?:aerial\s+|satellite\s+|grayscale\s+|color\s+|overhead\s+)?(?:image|view|photo|imagery|picture))", re.I), ""),
    (re.compile(r"\b(?:high|medium|low|moderate)[- ]resolution\s+(?=[a-z])", re.I), ""),
    (re.compile(r"\s+without\s+(?:a\s+)?(?:specified|stated|given|provided|specific)\s+resolution", re.I), ""),
    (re.compile(r"\s+with\s+(?:an?\s+)?(?:unspecified|unknown|no)\s+resolution", re.I), ""),
]

def clean_caption(text):
    out = text
    for rx, repl in _RULES:
        out = rx.sub(repl, out)
    out = re.sub(r"\s+([,.;:])", r"\1", out)
    out = re.sub(r",\s*,", ",", out)
    out = re.sub(r",\s*\.", ".", out)
    out = re.sub(r"\s{2,}", " ", out).strip()
    out = re.sub(r"\b([Aa]) (?=(?:aerial|overhead|urban|industrial|open|angled|oblique|elevated|image)\b)", lambda m: m.group(1) + "n ", out)
    out = re.sub(r"^(The|This) image,\s+(with|features|shows|displays|depicts|captures|showcases|presents|contains|includes)\b", lambda m: f"{m.group(1)} image {m.group(2)}", out)
    return out[:1].upper() + out[1:] if out else out

_BROKEN = re.compile(r"goog|sourced|provided by|courtesy|\bGF\b|\bJL\b|gaofen|jilin|the imagery|resolution|^(The|This) image (and|is|was|with|by)\b|\bby the\b\s*[.,]", re.I)
def usable(caption):
    return len(caption.split()) >= 12 and not _BROKEN.search(caption)

train_json = json.load(open(paths["VRSBench_train.json"]))
raw_captions = {it["image"]: it["conversations"][1]["value"] for it in train_json
                if it["conversations"][0]["value"].startswith("<image>\n[caption]")}
with zipfile.ZipFile(paths["Images_train.zip"]) as z:
    train_members = set(z.namelist())
entries = [(n, clean_caption(c)) for n, c in raw_captions.items() if f"Images_train/{n}" in train_members]
n_before = len(entries)
entries = [(n, c) for n, c in entries if usable(c)]
print(f"{len(raw_captions)} caption images, {n_before} with an image in the zip, {n_before - len(entries)} dropped as still-broken after cleaning")
random.Random(SEED).shuffle(entries)
val_entries, train_entries = entries[:N_VAL], entries[N_VAL:]
if SMOKE_TEST:
    train_entries = train_entries[:16]

eval_json = json.load(open(paths["VRSBench_EVAL_Cap.json"]))
with zipfile.ZipFile(paths["Images_val.zip"]) as z:
    eval_members = set(z.namelist())
test_pool = [(x["image_id"], clean_caption(x["ground_truth"])) for x in eval_json if f"Images_val/{x['image_id']}" in eval_members]
test_pool = [(n, c) for n, c in test_pool if usable(c)]  # the same still-broken filter as the training captions
test_entries = random.Random(SEED).sample(test_pool, min(N_TEST, len(test_pool)))

lens = [len(c.split()) for _, c in train_entries]
print(f"train {len(train_entries)} | val {len(val_entries)} | test {len(test_entries)} (of {len(test_pool)} eval images)")
print(f"train caption length: mean {np.mean(lens):.1f} words, max {max(lens)}")
scene = lambda n: re.match(r"(P?\d+)", n).group(1)
train_scenes = {scene(n) for n, _ in train_entries}
shared = sum(scene(n) in train_scenes for n, _ in test_entries) / len(test_entries)
print(f"test images whose scene id also occurs in train: {shared:.0%} (the dataset authors' own split -- DOTA crops of one scene can land on both sides; informational)")
for n, c in train_entries[:3]:
    print(" -", n, "|", c[:200])

train_images = ZipImages(paths["Images_train.zip"], "Images_train")
eval_images = ZipImages(paths["Images_val.zip"], "Images_val")

def make_loader(entries, images, shuffle):
    return DataLoader(Captions(entries, images), batch_size=BATCH, shuffle=shuffle, drop_last=shuffle, num_workers=WORKERS,
                      collate_fn=collate, persistent_workers=(WORKERS > 0))

train_loader = make_loader(train_entries, train_images, shuffle=True)
val_loader = make_loader(val_entries, train_images, shuffle=False)

probe = next(iter(val_loader))
supervised = int((probe["labels"] != -100).sum(1).float().mean())
print(f"batch tensors: input_ids {tuple(probe['input_ids'].shape)}, pixel_values {tuple(probe['pixel_values'].shape)}; supervised tokens per caption ~{supervised}")
assert 20 < supervised < 250, "answer-only masking looks wrong"
first = probe["labels"][0]
print("supervised text of the first caption:", tokenizer.decode(first[first != -100])[:160])

## 6. The zero-shot baseline

The same helper generates for the baseline and for the fine-tuned model (greedy, `repetition_penalty=1.05`, left padding), so the comparison differs only in the weights. (First real run, before the eval references were filtered: BLEU-4 0.022, ROUGE-L 0.198, CIDEr 0.001, and it wrote **106 words on average against the references' 46** -- a chatty generic describer, 569 s for 1,000 images on a T4.)

In [ ]:
def evaluate_model(model, tag):
    t0 = time.time()
    hyps = generate(model, test_entries, eval_images)
    scores = caption_metrics([c for _, c in test_entries], hyps)
    print(f"\n=== {tag}: {len(hyps)} test images, {time.time() - t0:.0f}s ===")
    print("  " + "  ".join(f"{k} {v:.3f}" for k, v in scores.items()))
    for (name, ref), hyp in list(zip(test_entries, hyps))[:3]:
        print(f"  [{name}]\n    REF: {ref[:230]}\n    GEN: {hyp[:230]}")
    return scores, hyps

model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.float32).to(DEVICE)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters (fp32 master weights; fp16 autocast does the fast math)")
zero_shot_scores, _ = evaluate_model(model, "ZERO-SHOT baseline (same prompt, same images)")

## 7. LoRA on the language model + a trainable connector

The vision encoder stays frozen. LoRA (r=32) goes on every attention and MLP projection of the *text* model only (the regex excludes the SigLIP vision layers, whose projections share the same names); the small connector that maps vision features into the language model's space is trained in full, at a quarter of the learning rate.

In [ ]:
model = add_lora(model)
lora_params = [p for n, p in model.named_parameters() if p.requires_grad and "lora_" in n]
connector_params = [p for n, p in model.named_parameters() if p.requires_grad and "connector" in n]
total = sum(p.numel() for p in model.parameters())
print(f"trainable: LoRA {sum(p.numel() for p in lora_params) / 1e6:.1f}M + connector {sum(p.numel() for p in connector_params) / 1e6:.1f}M of {total / 1e6:.0f}M total")
assert lora_params and connector_params

## 8. Train

AdamW, warm-up then cosine decay, fp16 autocast + gradient scaling on GPU. Validation loss (teacher-forced, on the 300 held-out training images) every epoch; the weights with the lowest validation loss are kept. A non-finite loss skips the step, and more than 5% of them aborts the run -- an fp16 overflow should stop the run early, not quietly poison it.

In [ ]:
optimizer = torch.optim.AdamW(
    [{"params": lora_params, "lr": LR}, {"params": connector_params, "lr": LR / 4}], weight_decay=0.01)
steps_total = EPOCHS * len(train_loader)
warmup = max(1, int(0.03 * steps_total))
def lr_factor(step):
    if step < warmup:
        return (step + 1) / warmup
    progress = (step - warmup) / max(1, steps_total - warmup)
    return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
use_amp = DEVICE == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

@torch.no_grad()
def validation_loss():
    model.eval()
    total, count = 0.0, 0
    for batch in val_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            n = int((batch["labels"] != -100).sum())
            total += float(model(**batch).loss) * n
        count += n
    return total / max(count, 1)

def trainable_state():
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}

CKPT_DIR = "/tmp/caption_ckpt" if SMOKE_TEST else "/kaggle/working/caption_ckpt"
os.makedirs(CKPT_DIR, exist_ok=True)
best_val, best_state, skipped, step = float("inf"), None, 0, 0
t_start = time.time()
print(f"{steps_total} steps ({len(train_loader)} per epoch), batch {BATCH}")
for epoch in range(EPOCHS):
    model.train()
    running, n_batches, t_epoch = 0.0, 0, time.time()
    for batch in train_loader:
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            loss = model(**batch).loss
        if not torch.isfinite(loss):
            skipped += 1
            optimizer.zero_grad(set_to_none=True)
            assert skipped <= max(3, 0.05 * (step + 1)), "too many non-finite losses -- fp16 is overflowing"
            step += 1
            scheduler.step()
            continue
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(lora_params + connector_params, 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        running += loss.item(); n_batches += 1; step += 1
        if step % 100 == 0:
            print(f"  step {step}/{steps_total}  loss {running / n_batches:.4f}  ({time.time() - t_start:.0f}s)", flush=True)
    val = validation_loss()
    marker = ""
    if val < best_val:
        best_val, best_state, marker = val, trainable_state(), "  <- best"
        torch.save(best_state, os.path.join(CKPT_DIR, "best_trainable.pt"))  # ~120MB; lets a failed export be redone without retraining
    print(f"epoch {epoch + 1}/{EPOCHS}  train loss {running / max(n_batches, 1):.4f}  val loss {val:.4f}  ({time.time() - t_epoch:.0f}s){marker}", flush=True)
    sample = generate(model, val_entries[:2], train_images, max_new_tokens=MAX_NEW_TOKENS)
    for (name, ref), hyp in zip(val_entries[:2], sample):
        print(f"    [{name}] GEN: {hyp[:200]}")
print(f"\ntraining took {(time.time() - t_start) / 60:.1f} min; best val loss {best_val:.4f}; skipped {skipped} non-finite steps")
model.load_state_dict(best_state, strict=False)

## 9. Test on VRSBench's official held-out eval images

In [ ]:
tuned_scores, tuned_hyps = evaluate_model(model, "FINE-TUNED (best-val weights)")
print("\nzero-shot -> fine-tuned:")
for k in tuned_scores:
    print(f"  {k:22s} {zero_shot_scores[k]:8.3f} -> {tuned_scores[k]:8.3f}")

## 10. Export

Merge the LoRA into the base weights and save a plain half-precision checkpoint directory (about 1GB) that `models/captioning/caption_tool.py` loads with `AutoModelForImageTextToText.from_pretrained` -- no PEFT needed at inference. Then reload the *saved artifact* and caption one test image, so a broken export fails here and not on the laptop.

In [ ]:
OUT_DIR = "/tmp/caption_model" if SMOKE_TEST else "/kaggle/working/caption_model"
os.makedirs(OUT_DIR, exist_ok=True)

merged = model.merge_and_unload().half()
merged.save_pretrained(OUT_DIR, safe_serialization=True)
processor.save_pretrained(OUT_DIR)
meta = {
    "base_model": MODEL_ID,
    "prompt": PROMPT,
    "do_image_splitting": False,
    "task": "single-image detailed caption (VRSBench, provenance boilerplate stripped)",
    "train_images": len(train_entries), "epochs": EPOCHS, "best_val_loss": best_val,
    "test_images": len(test_entries),
    "zero_shot_metrics": zero_shot_scores, "fine_tuned_metrics": tuned_scores,
}
json.dump(meta, open(os.path.join(OUT_DIR, "caption_meta.json"), "w"), indent=1)
size_mb = sum(os.path.getsize(os.path.join(OUT_DIR, f)) for f in os.listdir(OUT_DIR)) / 1e6
print(f"Exported to {OUT_DIR} ({size_mb:.0f} MB): {sorted(os.listdir(OUT_DIR))}")

del merged, model
torch.cuda.empty_cache() if DEVICE == "cuda" else None
reloaded = AutoModelForImageTextToText.from_pretrained(OUT_DIR, dtype=torch.float16).to(DEVICE)
check = generate(reloaded, test_entries[:2], eval_images, batch=2, max_new_tokens=MAX_NEW_TOKENS)
assert all(len(t.split()) >= 3 for t in check), f"reloaded artifact produced junk: {check}"
print("artifact round-trip OK:", check[0][:160])

## Next

Download `caption_model/` from the Output tab into `models/captioning/checkpoints/caption_model/` (gitignored), then check it on real scenes before wiring it anywhere: the metrics above measure caption *style*; whether the description is *true* for an image it has never seen -- a port, a farm field, a stadium -- needs looking at.